In [3]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

class AlphaLibrary:
    @staticmethod
    def momentum_factor(df, slow=20, fast=5):
        """Shortened for demo: Slow (20d) minus Fast (5d) momentum"""
        returns_slow = df.groupby(level=1)['close'].pct_change(slow)
        returns_fast = df.groupby(level=1)['close'].pct_change(fast)
        return returns_slow - returns_fast

    @staticmethod
    def volatility_factor(df, window=5):
        """Shortened for demo: 5-day rolling volatility"""
        returns = df.groupby(level=1)['close'].pct_change()
        return returns.groupby(level=1).rolling(window).std().reset_index(0, drop=True)

class QuantEngine:
    @staticmethod
    def preprocess(df):
        """Winsorize and Z-Score cross-sectionally."""
        def section_transform(x):
            if x.isnull().all(): return x
            # Winsorize 5% to 95% for small datasets
            x = x.clip(lower=x.quantile(0.05), upper=x.quantile(0.95))
            # Z-Score
            std = x.std()
            return (x - x.mean()) / std if std > 0 else x - x.mean()

        return df.groupby(level=0, group_keys=False).transform(section_transform)

    @staticmethod
    def calculate_ic(factor_series, return_series):
        """Calculates Daily Spearman Rank IC."""
        combined = pd.DataFrame({'factor': factor_series, 'ret': return_series}).dropna()
        if combined.empty:
            return pd.Series(dtype=float)

        # Use .corr(method='spearman') directly on the group
        return combined.groupby(level=0).apply(
            lambda x: x['factor'].corr(x['ret'], method='spearman') if len(x) > 1 else np.nan,
            include_groups=False
        )

# ==========================================
# EXECUTION PIPELINE
# ==========================================

# 1. Mock Data Generation (150 days to ensure window coverage)
dates = pd.date_range("2023-01-01", periods=150)
tickers = ['AAPL', 'MSFT', 'GOOG', 'AMZN', 'META', 'TSLA', 'NVDA']
index = pd.MultiIndex.from_product([dates, tickers], names=['date', 'ticker'])
data = pd.DataFrame(index=index)
np.random.seed(42)
data['close'] = np.random.uniform(100, 200, len(index)) * (1 + np.random.normal(0, 0.02, len(index)).cumsum())

# 2. Factor Generation
lib = AlphaLibrary()
factors = pd.DataFrame(index=data.index)
factors['momentum'] = lib.momentum_factor(data)
factors['volatility'] = lib.volatility_factor(data)

# 3. Preprocessing
engine = QuantEngine()
processed_factors = engine.preprocess(factors)

# 4. Target Calculation (Forward returns)
data['next_day_ret'] = data.groupby(level=1)['close'].pct_change().shift(-1)

# 5. Evaluation with check for empty results
ic_results = {}
for col in processed_factors.columns:
    daily_ic = engine.calculate_ic(processed_factors[col], data['next_day_ret'])
    if not daily_ic.empty:
        ic_results[col] = daily_ic

# 6. Final Output Summary
if ic_results:
    ic_df = pd.DataFrame(ic_results)
    print("--- Alpha Factor Performance (Mean IC) ---")
    print(ic_df.mean())
    print("\n--- IC Information Ratio ---")
    print(ic_df.mean() / ic_df.std())
else:
    print("No IC calculated. Check if windows are too large for the dataset.")

--- Alpha Factor Performance (Mean IC) ---
momentum      0.010000
volatility   -0.041232
dtype: float64

--- IC Information Ratio ---
momentum      0.026300
volatility   -0.093227
dtype: float64
